# SAM3 masking 파이프라인 — worker 실제 사진 재검증

**목적**: `AI/validation/QUAL/sam3-condition-check/sam3_qc_test_masking.ipynb`가 쓰는
탐지·필터·중복병합 로직을 **그대로**(재구현 아님 — `qc_shared.py`를 그대로 import) 가져와서,
`worker/validation/sample_images/`의 **실제 상품 사진**으로 다시 돌려본다.

**배경**: 그 스파이크 노트북은 SAM3를 QC 백엔드로 채택할지 판단하려고 먼저 만들어진
것이고(프로덕션보다 이전), VLM 요약 성능 검증 때는 `small_pro.jpg`에서 6건이 나왔다.
그런데 지금 프로덕션 경로(`quality_check_e2e_validation.ipynb` → 실제
`worker/models/sam3/`)의 같은 이미지 결과는 다르다 — cross-type 중복 병합(`dedupe_cross_type`)
차이만으로는 설명이 안 될 정도로 달라서, **탐지 자체가 다른 로직으로 갈렸는지** 이 노트북으로
확인한다.

**주의**: 이 노트북은 스파이크의 로직을 재사용할 뿐, 프로덕션 코드(`worker/models/sam3/`)를
건드리거나 대체하지 않는다. 결과를 비교해 실제로 프로덕션 로직을 바꿔야 한다고 판단되면
그건 별도 설계 논의(계획 문서)가 필요하다.

## 실행 방법

- SAM3(`facebook/sam3`)는 **gated repo** — 약관 동의한 HF 계정으로 로그인 필요.
- `transformers>=5.0` 필요 — `worker/validation/quality_check_e2e_validation.ipynb`와
  같은 conda 환경(`relabel`/`qual` 등, transformers>=5 계열)에서 돌리면 된다.
- 이 노트북도 `worker/` 또는 `worker/validation/` 어디서 열어도 되게 cwd를 자동 감지한다.
- `qc_shared.py`(`AI/validation/QUAL/`)를 그대로 import한다 — 필터·중복병합 로직을 한 글자도
  다시 안 짠다. SAM3 모델 호출 글루 코드(`detect`/`inspect_image` 등)만 이 노트북에 있다
  (스파이크 노트북들도 원래 이 부분만 노트북마다 따로 갖고 있다 — 백엔드별 모델 호출이라
  공유 모듈에 안 두는 설계).


In [ ]:
# 의존성 설치 — SAM3는 transformers>=5.0 필요(4.57.x 이하엔 Sam3Model이 없음).
# quality_check_e2e_validation.ipynb와 같은 계열 패키지라 같은 conda 환경에서 돌려도 된다.
# CUDA 휠 인덱스는 환경마다 다르므로(로컬 RTX 4070 vs GPU 서버) 여기서 강제하지 않는다 —
# 필요하면 이 셀 실행 전에 맞는 --extra-index-url로 torch를 먼저 깔아 둘 것.
%pip install -q "torch>=2.4" "transformers>=5.0" "accelerate>=1.0" "huggingface_hub>=0.34" \
    "opencv-python>=4.10" "Pillow>=10.0" "numpy>=1.26" "pandas>=2.2" "matplotlib>=3.9" "requests>=2.32"
print("설치 완료")


## 0-1. Hugging Face 로그인 (gated repo)

`facebook/sam3`는 gated(수동 승인) 저장소다. **모델 페이지에서 약관에 동의한 계정**으로
로그인해야 가중치를 받을 수 있다.


In [ ]:
import os

from huggingface_hub import login, whoami

_token = os.environ.get("HF_TOKEN")
if _token:
    login(token=_token, add_to_git_credential=False)
    print("HF_TOKEN 환경변수로 로그인했습니다.")
else:
    try:
        from huggingface_hub import notebook_login

        notebook_login()
    except Exception as exc:
        print(f"[주의] notebook_login()을 띄우지 못했습니다: {type(exc).__name__}: {exc}")
        print("아래 중 하나로 처리하세요:")
        print('  1) 이 셀에서 직접:  login(token="hf_xxxxxxxx")')
        print("  2) 터미널에서:      huggingface-cli login")
        print("  3) 환경변수 HF_TOKEN을 세팅하고 커널 재시작")

try:
    _me = whoami()
    print(f"로그인 계정: {_me.get('name')}")
except Exception as exc:
    print(f"[주의] 아직 로그인 상태가 아닙니다: {type(exc).__name__}: {exc}")
    print("facebook/sam3는 gated repo라 로그인 없이는 다음 셀들의 모델 로드가 실패합니다.")


## 0-2. 경로 설정 + CUDA 확인

`quality_check_e2e_validation.ipynb`와 같은 방식 — cwd가 `worker/`든 `worker/validation/`든
자동으로 맞춘다. `qc_shared.py`는 `AI/validation/QUAL/`에서 그대로 가져온다(재구현 없음).


In [ ]:
import sys
import torch
from pathlib import Path

_CWD = Path(os.getcwd())
_WORKER_DIR = _CWD if (_CWD / "models").is_dir() else _CWD.parent
_REPO_ROOT = _WORKER_DIR.parent
assert (_WORKER_DIR / "models").is_dir(), f"'{_WORKER_DIR}'가 worker/가 아닌 것 같다"

# qc_shared.py(AI/validation/QUAL/)를 그대로 재사용 — 필터·중복병합 로직을 다시 안 짠다.
_QC_SHARED_DIR = _REPO_ROOT / "AI" / "validation" / "QUAL"
assert _QC_SHARED_DIR.is_dir(), f"'{_QC_SHARED_DIR}'가 없다 — 리포 구조가 예상과 다르다"
sys.path.insert(0, str(_QC_SHARED_DIR))

from qc_shared import (  # noqa: E402
    BACKEND_THRESHOLDS,
    BENIGN_CONTAINMENT,
    BENIGN_PROMPTS,
    CROSS_TYPE_IOU_MIN,
    DEFECT_PROMPTS,
    GARMENT_OVERLAP_MIN,
    GARMENT_PROMPT,
    KOREAN_LABELS,
    RESTRICT_TO_GARMENT,
    SUPPRESS_BENIGN,
    VERDICT_THRESHOLDS,
    decide_verdict,
    dedupe_cross_type,
    draw_numbered_boxes,
    filter_by_garment,
    list_images,
    pad_bbox_clamped,
    pick_largest,
    remap_instances_to_full,
    remap_mask_to_full,
    renumber,
    suppress_benign,
    to_serializable,
    use_korean_font,
)

MODEL_ID = "facebook/sam3"
DTYPE = torch.float16
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 실제 상품 사진 — worker/validation/sample_images/(로컬 전용, .gitignore)
IMAGE_DIR = _WORKER_DIR / "validation" / "sample_images"
OUTPUT_DIR = _WORKER_DIR / "validation" / "outputs" / "sam3_masking_check"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SCORE_THRESHOLD = BACKEND_THRESHOLDS["sam3"]["score"]
GARMENT_SCORE_THRESHOLD = BACKEND_THRESHOLDS["sam3"]["garment_score"]
MASK_THRESHOLD = 0.5
GARMENT_CROP_PAD_RATIO = 0.06  # sam3_qc_test_masking.ipynb와 같은 값

print(f"WORKER_DIR   = {_WORKER_DIR}")
print(f"IMAGE_DIR    = {IMAGE_DIR}")
print(f"프롬프트     : {DEFECT_PROMPTS}")
print(f"의류 게이트  : {RESTRICT_TO_GARMENT} (겹침 최소 {GARMENT_OVERLAP_MIN})")
print(f"양성 억제    : {SUPPRESS_BENIGN} (겹침 최소 {BENIGN_CONTAINMENT})")
print(f"cross-type dedup IoU 최소 : {CROSS_TYPE_IOU_MIN}")

print()
print(f"torch          : {torch.__version__}")
print(f"cuda available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    _props = torch.cuda.get_device_properties(0)
    print(f"gpu            : {_props.name}")
    print(f"total VRAM     : {_props.total_memory / 1024**3:.2f} GB")
else:
    print("[주의] CUDA를 찾지 못했다 — SAM3 1008px 추론은 CPU에서 매우 느리다.")


## 0-3. 모델 로드 — `Sam3Model` + `Sam3Processor`

`torch_dtype=`은 transformers v5에서 deprecated라 `dtype=`을 쓴다.


In [ ]:
import transformers
from transformers import Sam3Model, Sam3Processor

print(f"transformers: {transformers.__version__} (SAM3는 >=5.0 필요)")

if DEVICE == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

try:
    processor = Sam3Processor.from_pretrained(MODEL_ID)
    model = Sam3Model.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE).eval()
    print(f"로드 완료: {MODEL_ID}  dtype={DTYPE}  device={DEVICE}")
except Exception as exc:
    _msg = str(exc).lower()
    if any(k in _msg for k in ("gated", "401", "403", "awaiting", "access to model")):
        print("[게이트 접근 실패] facebook/sam3는 gated repo다.")
        print("  1) https://huggingface.co/facebook/sam3 에서 약관 동의")
        print("  2) 0-1 셀에서 로그인한 계정이 그 계정인지 확인")
    else:
        print(f"[모델 로드 실패] {type(exc).__name__}: {exc}")
    raise

if DEVICE == "cuda":
    _alloc = torch.cuda.max_memory_allocated() / 1024**2
    print(f"로드 직후 VRAM 사용량: {_alloc:.1f} MB")


## 0-4. 테스트 이미지 로드

`IMAGE_DIR`(`worker/validation/sample_images/`)의 이미지를 전부 스캔한다.


In [ ]:
from PIL import Image


def load_image(path):
    return Image.open(path).convert("RGB")


IMAGE_PATHS = list_images(IMAGE_DIR)
print(f"발견된 이미지 {len(IMAGE_PATHS)}장:")
for _i, _p in enumerate(IMAGE_PATHS):
    print(f"  [{_i}] {_p.name}")

# small_pro.jpg를 기준 이미지로 잡는다(VLM 검증 때 6건 나왔던 그 이미지 — 비교 대상).
_small_pro = [p for p in IMAGE_PATHS if p.name == "small_pro.jpg"]
TARGET_INDEX = IMAGE_PATHS.index(_small_pro[0]) if _small_pro else 0
IMAGE_PATH = IMAGE_PATHS[TARGET_INDEX]
print()
print(f"기준 이미지: [{TARGET_INDEX}] {IMAGE_PATH.name}")


## 1. 탐지 파이프라인 — `sam3_qc_test_masking.ipynb`와 동일

의류 크롭(+마스크 밖 검정 처리) → 크롭 안에서 하자·양성 탐지 → 원본 좌표로 복원 → 의류
실루엣 재확인 → 양성 억제 → **cross-type 중복 병합**까지, 그 스파이크 노트북의
`inspect_image()`를 그대로 옮겼다(함수 이름·순서·인자 동일). 필터·병합 로직 자체
(`filter_by_garment`/`suppress_benign`/`dedupe_cross_type`)는 옮겨적은 게 아니라
`qc_shared`에서 그대로 import한 것이라 한 글자도 다르지 않다.


In [ ]:
import numpy as np


def _to_device(inputs):
    inputs = inputs.to(DEVICE)
    if "pixel_values" in inputs and inputs["pixel_values"].dtype.is_floating_point:
        inputs["pixel_values"] = inputs["pixel_values"].to(DTYPE)
    return inputs


def _post_process(outputs, original_sizes, score_threshold=SCORE_THRESHOLD):
    return processor.post_process_instance_segmentation(
        outputs,
        threshold=score_threshold,
        mask_threshold=MASK_THRESHOLD,
        target_sizes=original_sizes,
    )[0]


def detect(pil_image, prompt, *, vision_embeds=None, score_threshold=SCORE_THRESHOLD):
    if vision_embeds is not None:
        try:
            text_inputs = _to_device(processor(text=prompt, return_tensors="pt"))
            with torch.no_grad():
                outputs = model(vision_embeds=vision_embeds, **text_inputs)
            sizes = [[pil_image.height, pil_image.width]]
            return _post_process(outputs, sizes, score_threshold), outputs
        except Exception as exc:
            print(f"  [폴백] vision_embeds 재사용 실패({type(exc).__name__}: {exc}) -> 전체 호출")

    inputs = _to_device(processor(images=pil_image, text=prompt, return_tensors="pt"))
    with torch.no_grad():
        outputs = model(**inputs)
    return _post_process(outputs, inputs["original_sizes"].tolist(), score_threshold), outputs


def detect_all(pil_image, prompts=DEFECT_PROMPTS, score_threshold=SCORE_THRESHOLD, vision_embeds=None):
    merged = []
    for defect_type, prompt in prompts.items():
        res, _ = detect(pil_image, prompt, vision_embeds=vision_embeds, score_threshold=score_threshold)
        for score, box, mask in zip(res["scores"].tolist(), res["boxes"].tolist(), res["masks"]):
            merged.append(
                {
                    "type": defect_type,
                    "prompt": prompt,
                    "bbox": [int(round(v)) for v in box],
                    "confidence": round(float(score), 4),
                    "mask": mask.detach().cpu().numpy().astype(np.uint8),
                }
            )
    return renumber(merged)


def vision_embeds_for(pil_image):
    try:
        img_inputs = _to_device(processor(images=pil_image, return_tensors="pt"))
        with torch.no_grad():
            return model.get_vision_features(pixel_values=img_inputs["pixel_values"])
    except Exception as exc:
        print(f"  [폴백] get_vision_features 실패({type(exc).__name__}: {exc}) -> 매번 전체 호출")
        return None


def detect_garment(pil_image, vision_embeds=None):
    res, _ = detect(pil_image, GARMENT_PROMPT, vision_embeds=vision_embeds, score_threshold=GARMENT_SCORE_THRESHOLD)
    found = []
    for score, box, mask in zip(res["scores"].tolist(), res["boxes"].tolist(), res["masks"]):
        found.append(
            {
                "type": "GARMENT",
                "confidence": round(float(score), 4),
                "bbox": [int(round(v)) for v in box],
                "mask": mask.detach().cpu().numpy().astype(np.uint8),
            }
        )
    return found


def crop_to_garment(pil_image):
    picked = pick_largest(detect_garment(pil_image))
    if picked is None:
        return pil_image, None, None

    crop_box = pad_bbox_clamped(picked["bbox"], GARMENT_CROP_PAD_RATIO, pil_image.size)
    x1, y1, x2, y2 = crop_box
    crop_image = pil_image.crop((x1, y1, x2, y2))
    garment_mask_in_crop = picked["mask"][y1:y2, x1:x2]

    arr = np.array(crop_image)
    arr[garment_mask_in_crop == 0] = 0
    crop_image = Image.fromarray(arr)

    return crop_image, crop_box, garment_mask_in_crop


def detect_benign(pil_image, vision_embeds=None):
    found = []
    for tag, prompt in BENIGN_PROMPTS.items():
        res, _ = detect(pil_image, prompt, vision_embeds=vision_embeds)
        for score, box, mask in zip(res["scores"].tolist(), res["boxes"].tolist(), res["masks"]):
            found.append(
                {
                    "type": tag,
                    "confidence": round(float(score), 4),
                    "bbox": [int(round(v)) for v in box],
                    "mask": mask.detach().cpu().numpy().astype(np.uint8),
                }
            )
    return found


def inspect_image(pil_image):
    full_size = pil_image.size
    crop_image, crop_box, garment_mask_in_crop = (
        crop_to_garment(pil_image) if RESTRICT_TO_GARMENT else (pil_image, None, None)
    )
    cropped = crop_box is not None
    detect_image = crop_image if cropped else pil_image

    embeds = vision_embeds_for(detect_image)
    raw_in_crop = detect_all(detect_image, prompts=DEFECT_PROMPTS, vision_embeds=embeds)
    benigns_in_crop = detect_benign(detect_image, embeds) if SUPPRESS_BENIGN else []

    if cropped:
        raw = remap_instances_to_full(raw_in_crop, crop_box, full_size)
        benigns = remap_instances_to_full(benigns_in_crop, crop_box, full_size)
        garment_mask = remap_mask_to_full(garment_mask_in_crop, crop_box, full_size)
    else:
        raw, benigns, garment_mask = raw_in_crop, benigns_in_crop, None

    after_garment, off_garment = filter_by_garment(raw, garment_mask, GARMENT_OVERLAP_MIN)
    final, on_benign = suppress_benign(after_garment, benigns, BENIGN_CONTAINMENT)
    final, cross_type_dupes = dedupe_cross_type(final, CROSS_TYPE_IOU_MIN)

    trace = {
        "raw": len(raw),
        "off_garment": len(off_garment),
        "on_benign": len(on_benign),
        "cross_type_dupes": len(cross_type_dupes),
        "final": len(final),
        "garment_found": cropped,
    }
    return renumber(final), trace


## 2. 전체 이미지 실행 — 단계별 필터링 트레이스

각 단계에서 몇 건이 걷어졌는지 표로 본다. `small_pro.jpg` 행을 프로덕션(e2e 노트북) 결과와
바로 대조할 것.


In [ ]:
import pandas as pd

results_by_image = {}
traces_by_image = {}
for _path in IMAGE_PATHS:
    _final, _trace = inspect_image(load_image(_path))
    results_by_image[_path] = _final
    traces_by_image[_path] = _trace
    _mark = "  <- 비교 대상" if _path.name == "small_pro.jpg" else ""
    print(
        f"  {_path.name:<24} raw {_trace['raw']:>3} "
        f"-> 배경 제외 -{_trace['off_garment']:<3} "
        f"-> 주름·그림자 억제 -{_trace['on_benign']:<3} "
        f"-> 타입간 중복 병합 -{_trace['cross_type_dupes']:<3} "
        f"=> {_trace['final']:>3}건{_mark}"
    )

_rows = [
    {"image": p.name, **t}
    for p, t in traces_by_image.items()
]
trace_df = pd.DataFrame(_rows)
trace_df


## 3. 넘버링 + bbox 시각화

`qc_shared.draw_numbered_boxes()` — 스파이크 노트북과 완전히 같은 함수.


In [ ]:
import matplotlib.pyplot as plt

print(f"matplotlib 한글 폰트: {use_korean_font()}")

_n_img = len(results_by_image)
fig, axes = plt.subplots(_n_img, 1, figsize=(10, 8.5 * _n_img), squeeze=False)

for _row, (_path, _insts) in enumerate(results_by_image.items()):
    _annotated = draw_numbered_boxes(load_image(_path), _insts)
    _out = OUTPUT_DIR / f"{_path.stem}_masking.png"
    Image.fromarray(_annotated).save(_out)
    axes[_row][0].imshow(_annotated)
    axes[_row][0].set_title(f"[{_row}] {_path.name} — {len(_insts)} instances")
    axes[_row][0].axis("off")

plt.tight_layout()
plt.show()
print(f"bbox 이미지 {_n_img}장을 {OUTPUT_DIR}에 저장했습니다.")


## 4. 하자 요약 + 판정 — 프로덕션 결과와 비교용

인스턴스 목록(JSON 직렬화 가능)과, 한글 라벨 기준 개수 요약을 만든다. `small_pro.jpg`의
`final` 건수·유형별 개수를 e2e 노트북(`quality_check_e2e_validation.ipynb`)의
`result['findings']`/`result['image_summaries']`와 나란히 놓고 비교할 것.


In [ ]:
import json


def summarize_instances(insts):
    """[{"no","type","label","confidence","bbox"}, ...] 텍스트 요약 한 줄씩."""
    if not insts:
        return "확인된 하자가 없습니다."
    lines = []
    for inst in sorted(insts, key=lambda d: d["no"]):
        label = KOREAN_LABELS.get(inst["type"], inst["type"])
        lines.append(f"{inst['no']}. {label} — confidence {inst['confidence']:.3f}, bbox={inst['bbox']}")
    return "\n".join(lines)


qc_results = {}
for _path in IMAGE_PATHS:
    _insts = to_serializable(results_by_image[_path])
    _verdict, _peak = decide_verdict(results_by_image[_path], VERDICT_THRESHOLDS)
    _type_counts = {}
    for _inst in _insts:
        _label = KOREAN_LABELS.get(_inst["type"], _inst["type"])
        _type_counts[_label] = _type_counts.get(_label, 0) + 1
    qc_results[_path] = {
        "instances": _insts,
        "verdict": _verdict,
        "peak_confidence": round(_peak, 4),
        "type_counts": _type_counts,
        "summary": summarize_instances(_insts),
    }
    (OUTPUT_DIR / f"{_path.stem}_qc.json").write_text(
        json.dumps(
            {"image": _path.name, **qc_results[_path]}, ensure_ascii=False, indent=2
        ),
        encoding="utf-8",
    )

print("=== 전체 이미지 하자 요약 ===")
for _path, _r in qc_results.items():
    _mark = "  <- 비교 대상" if _path.name == "small_pro.jpg" else ""
    print(f"\n[{_path.name}] verdict={_r['verdict']} (peak_conf={_r['peak_confidence']}){_mark}")
    print(f"  유형별: {_r['type_counts']}")
    print(f"  {_r['summary']}")

print()
print(f"저장 완료 -> {OUTPUT_DIR}")


## 5. 비교 (실행 후 채울 것)

| 항목 | 이 노트북(masking 스파이크) | 프로덕션(e2e 노트북) | 차이 |
| --- | --- | --- | --- |
| `small_pro.jpg` 총 건수 | | | |
| 유형별 개수 | | | |
| cross-type 중복 병합 건수 | | | |

차이가 여전히 크다면 다음을 의심할 것:
- 프로덕션(`worker/models/sam3/segmenter.py`/`findings.py`)이 이 스파이크와 **다른 프롬프트
  세트**(`ACTIVE_PROMPT_SET`)나 **다른 임계치**를 쓰고 있는지
- 프로덕션이 크롭+마스킹(`crop_to_garment`처럼 배경을 검정 처리) 단계를 아예 안 쓰고 있는지
  (즉 배경 오탐 억제 방식 자체가 다른지)
- benign 억제(`SUPPRESS_BENIGN`) 대상 프롬프트 목록이 프로덕션에도 동일하게 있는지

이 노트북은 진단용이라, 실제로 프로덕션 로직을 이 방식으로 바꿔야 한다는 결론이 나오면
그건 별도 설계 논의(계획 문서)로 넘길 것 — 여기서 바로 프로덕션 코드를 고치지 않는다.
